# Yang, Murray & Wang 2016 — every reproducible figure, from our model

This notebook goes through **every figure of Yang et al. 2016** (Nat. Commun. 7:12815) and its
supplement. Each one is either reproduced with **this project's model** or marked not reproducible,
with the reason.

**Where the numbers come from**

| part | how it is produced |
|---|---|
| spiking panels (Figs. 2, 3, 7c–e; Supp. Figs. 1, 3, 4) | `experiments/yang_figures.py`, run on toothless → `results/current/yang_figures.json`. The model is `model/parameters.py` + `model/equations.py` with the canonical synapse (NMDA only, w = 3ρ). Fig. 7 uses the full canonical circuit (`model/context_circuit.py`). |
| rate-network panels (Figs. 3–6; Supp. Figs. 5, 6) | Computed live in this notebook (`plotting/yang_rate_model.py`). Yang also produced these with a multi-compartment **rate model fitted to his spiking neuron**. Here that rate model is **refit to our spiking neuron** (Supp. Fig. 4 below). The interneuron networks follow Yang's Methods exactly. |
| analytic panels (Supp. Fig. 2c,d) | Yang's Eq. 24 with our constants. |
| Fig. 7b | `experiments/paper_comparison.py` (canonical) → `results/current/paper_comparison_summary.json`. |

In the rate figures, the dashed grey line is the same computation done with **Yang's published fit**.
It checks that the network code is implemented correctly, and it shows how far our neuron is from Yang's.

**Not reproducible with this model:**
- **Fig. 1**: a schematic; our circuit diagram is shown instead.
- **Fig. 8 and Supp. Fig. 8**: fits to monkey behaviour (Mante et al. 2013) through a separate decision network. They need the behavioural data and a model we do not have.
- **Supp. Fig. 7**: a fit to Sjöström et al. 2001 slice data.
- **Supp. Fig. 1a–d**: a reconstructed NEURON morphology.
- **Supp. Fig. 2a,b,e–h**: these need a non-saturating NMDA variant. Fig. 2c already covers low-rate inhibition.

Figures are saved to `figures/yang_replication/`.

In [ ]:
%matplotlib inline
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
PROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path[:0] = [os.path.join(PROOT, 'plotting'), os.path.join(PROOT, 'model'), PROOT]
import yang_rate_model as R
D = json.load(open(os.path.join(PROOT, 'results', 'current', 'yang_figures.json')))
FIG = os.path.join(PROOT, 'figures', 'yang_replication'); os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110,
                     'axes.titlesize': 10, 'axes.labelsize': 9, 'legend.fontsize': 8, 'legend.frameon': False})
LIGHT, DARK, ORANGE, GREY = '#6BAED6', '#08306B', '#C2510B', '#9A9A9A'
def save(fig, name):
    fig.savefig(os.path.join(FIG, name + '.png'), dpi=150, bbox_inches='tight', facecolor='white')
SUMMARY = []   # rows: (figure, Yang, ours, verdict)
print('loaded', len(D), 'result entries')

## Fig. 1 — the dendritic disinhibitory motif (schematic)

Yang's Fig. 1 is a diagram: control → VIP ⊣ SOM ⊣ pyramidal dendrite, with different pathways
on different dendrites. There is nothing to simulate. Here is our canonical circuit, which implements
the motif with MD as the control input (Zhang 2025 wiring):

In [ ]:
p = os.path.join(PROOT, 'figures', 'circuit_canonical.png')
display(Image(filename=p, width=700)) if os.path.exists(p) else print('figures/circuit_canonical.png not found')

## Fig. 2b,c,e–g — NMDA plateau, disinhibition opens the gate, tuning is transmitted

Yang's protocol:
- **b**: number of synapses activated together;
- **c**: input rate with the branch inhibited (35 Hz) or disinhibited (5 Hz);
- **e–g**: two pathways on two dendrites each, with input u = 40 exp(−z²) Hz.

Our neuron: `plotting/yang_fig2_supp1.py` with its defaults (in-vivo coupling 0.8 nS, soma background 300 Hz).

In [ ]:
f2 = D['fig2']['fig2']
fig = plt.figure(figsize=(14, 7.5)); gs = fig.add_gridspec(2, 3)
a = fig.add_subplot(gs[0, 0]); cm = plt.cm.Blues(np.linspace(0.35, 1, len(f2['volley_n'])))
for c, k, v in zip(cm, f2['volley_n'], f2['b_V']): a.plot(f2['b_t'], v, color=c, lw=1.3, label=str(k))
a.set(xlabel='time (ms)', ylabel='$V_D$ (mV)', title='b  NMDA plateau vs # synapses'); a.legend(title='# synapses', ncol=2)
a = fig.add_subplot(gs[0, 1])
a.plot(f2['rates'], f2['c'][1], 'o-', color=LIGHT, label='disinhibited (5 Hz)')
a.plot(f2['rates'], f2['c'][0], 'o-', color=DARK, label='inhibited (35 Hz)')
a.set(xlabel='excitatory input rate (Hz)', ylabel='mean $V_D$ (mV)', ylim=(-75, 0), title='c  disinhibition opens the gate'); a.legend()
a = fig.add_subplot(gs[0, 2])
a.plot(f2['z'], f2['e1'], 'o-', color=LIGHT, label='gate 1 open'); a.plot(f2['z'], f2['e2'], 'o-', color=DARK, label='gate 2 open')
a.set(xlabel='pathway 1 stimulus z', ylabel='output rate (Hz)', title=f"e  tuning to pathway 1 (S = {f2['S']:.2f})"); a.legend()
vmax = max(np.max(f2['fg_none']), np.max(f2['fg_1']))
for col, key, t in ((0, 'fg_none', 'f  2-D tuning, all gates closed'), (1, 'fg_1', 'g  2-D tuning, gate 1 open')):
    a = fig.add_subplot(gs[1, col])
    im = a.imshow(np.array(f2[key]).T, origin='lower', extent=[-2.4, 2.4, -2.4, 2.4], vmin=0, vmax=vmax, cmap='viridis', aspect='auto')
    a.set(xlabel='pathway 1 stimulus', ylabel='pathway 2 stimulus', title=t); fig.colorbar(im, ax=a, label='rate (Hz)')
fig.suptitle('Yang Fig. 2b,c,e–g — our spiking neuron', fontsize=12); fig.tight_layout(); save(fig, 'fig2'); plt.show()
SUMMARY.append(('Fig. 2e', 'S ≈ 0.6 (strong vs weak tuning)', f"S = {f2['S']:.2f}", 'match' if 0.4 < f2['S'] < 0.8 else 'check'))

## Supp. Fig. 1f–h — inhibition gates NMDA input strongly, AMPA input weakly

In [ ]:
s1 = D['fig2']['supp1']
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
cm = plt.cm.Blues(np.linspace(0.35, 1, len(s1['volley_n'])))
for c, k, v in zip(cm, s1['volley_n'], s1['f_V']): ax[0].plot(s1['f_t'], v, color=c, lw=1.3, label=str(k))
ax[0].set(xlabel='time (ms)', ylabel='$V_D$ (mV)', title='f  plateau vs # synapses'); ax[0].legend(title='# synapses', ncol=2)
for a, k, t in ((ax[1], 'g', 'g  NMDA input'), (ax[2], 'h', 'h  AMPA input')):
    a.plot(s1['rates'], s1[k][0], 'o-', color=LIGHT, label='no inhibition'); a.plot(s1['rates'], s1[k][1], 'o-', color=DARK, label='inhibition 30 Hz')
    a.set(xlabel='excitatory input rate (Hz)', ylabel='mean $V_D$ (mV)', ylim=(-75, 0), title=t); a.legend()
dN = np.max(np.array(s1['g'][0]) - np.array(s1['g'][1])); dA = np.max(np.array(s1['h'][0]) - np.array(s1['h'][1]))
fig.suptitle(f'Yang Supp. Fig. 1f–h — max. inhibition effect: NMDA {dN:.0f} mV, AMPA {dA:.0f} mV', fontsize=11)
fig.tight_layout(); save(fig, 'supp1_fgh'); plt.show()
SUMMARY.append(('Supp. 1g,h', 'inhibition gates NMDA ≫ AMPA', f'NMDA {dN:.0f} mV vs AMPA {dA:.0f} mV', 'match' if dN > 2 * dA else 'check'))

## Supp. Fig. 2c,d — why NMDA saturation lets mild inhibition control the plateau (analytic)

Yang's Eq. 24: with constant conductances, the steady state of one branch depends only on the
effective input x = g_NMDA / (g_L + g_GABA), because E_GABA = E_L. Above a threshold θ_NMDA, the low
(no-plateau) state disappears. Because NMDA saturates (Eq. 25), x has a ceiling, and moderate
inhibition keeps that ceiling below θ_NMDA. Our constants: g_L,D = 4 nS, g̃_NMDA = 2.5 nS,
15 synapses, our Mg block and NMDA kinetics.

In [ ]:
xs = np.linspace(0.01, 6, 600); lo, hi = [], []
for x in xs:
    v = R.vss_branches(x); lo.append(v.min() if len(v) else np.nan); hi.append(v.max() if len(v) else np.nan)
lo, hi = np.array(lo), np.array(hi)
theta = xs[np.argmax(lo > -40)] if np.any(lo > -40) else np.nan
from parameters import params
from brian2 import second
a_, tr_, td_ = float(params['alphaNMDA'] * second), float(params['tauNMDARise'] / second), float(params['tauNMDADecay'] / second)
sbar = lambda r: 1 - 1 / (1 + r * tr_ * td_ * a_)
rates = np.linspace(0, 200, 400)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(xs, lo, color=DARK, label='low state'); ax[0].plot(xs, hi, color=LIGHT, label='high (plateau) state')
ax[0].axvline(theta, ls='--', color=GREY, label=f'θ_NMDA ≈ {theta:.2f}')
ax[0].set(xlabel='effective input  g_NMDA / (g_L + g_GABA)', ylabel='steady-state $V_{ss}$ (mV)', title='c  steady states of one branch'); ax[0].legend()
for gI, c in ((0, LIGHT), (2, '#3182BD'), (4, DARK), (8, 'k')):
    ax[1].plot(rates, 15 * 2.5 * sbar(rates) / (4 + gI), color=c, label=f'g_GABA = {gI} nS')
ax[1].axhline(theta, ls='--', color=GREY); ax[1].set(xlabel='NMDA input rate (Hz, 15 synapses × 2.5 nS)', ylabel='effective input', title='d  saturation caps the effective input'); ax[1].legend()
fig.suptitle('Yang Supp. Fig. 2c,d — our constants', fontsize=11); fig.tight_layout(); save(fig, 'supp2_cd'); plt.show()
cap = {gI: 15 * 2.5 / (4 + gI) for gI in (0, 2, 4, 8)}
print('effective-input ceiling (saturated NMDA):', {k: round(v, 2) for k, v in cap.items()}, ' threshold', round(theta, 2))
SUMMARY.append(('Supp. 2c,d', 'moderate inhibition keeps saturated input below θ_NMDA', f'θ ≈ {theta:.2f}; ceiling at g_GABA 4/8 nS = {cap[4]:.2f}/{cap[8]:.2f}',
                'match' if cap[8] < theta < cap[0] else 'check'))

## Supp. Fig. 3c — AMPA input degrades gating

Fig. 2e protocol (S at the preferred stimulus, gate 1 vs gate 2 open). The total conductance per synapse is
fixed at 2.5 nS and split between AMPA and NMDA. Yang: S falls as the AMPA fraction rises.
The GABA_B panels (d, e) are not reproduced: our dendritic GABA_B is not part of the canonical model.

In [ ]:
af = sorted(float(k.split('_')[1]) for k in D if k.startswith('supp3_'))
S3 = [D[f'supp3_{a}']['S'] for a in af]
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.plot(af, S3, 'o-', color=DARK); ax.set(xlabel='AMPA fraction  g_AMPA / (g_AMPA + g_NMDA)', ylabel='gating selectivity S', ylim=(0, 1), title='Yang Supp. Fig. 3c — our neuron')
fig.tight_layout(); save(fig, 'supp3c'); plt.show()
print('S by AMPA fraction:', dict(zip(af, np.round(S3, 2))))
SUMMARY.append(('Supp. 3c', 'S decreases with AMPA fraction', f'S {S3[0]:.2f} (NMDA only) → {S3[-1]:.2f} (AMPA only)', 'match' if S3[0] > S3[-1] + 0.2 else 'check'))

## Supp. Fig. 4 — the rate model, refit to our spiking neuron

**Dendrite (b):**
- One branch with 15 NMDA synapses at 30 Hz. The per-synapse conductance is 0.25–5 nS, so g_E ≈ 2.4–48 nS (Eq. 25).
- One 4 nS GABA synapse at 0–100 Hz (g_I = r·τ·g, 0–8 nS).
- Soma clamped at −60 mV (Yang's Methods).

Fit: Yang's sigmoid family, Eqs. 20–22, re-referenced to rest (see `yang_rate_model.py`).

**Soma (c):** the dendritic voltage is held at V_D, and the input current is I = G_c (V_D − E_reset) with G_c = 8 nS.
Fit: the power law r = [max(0, I − I₀)/c]^p. The soma background (AMPA Poisson rate) is chosen so that
the rest rate matches Yang's (~1.8 Hz). The same background is used for the spiking Fig. 3.

In [ ]:
dd = D['supp4_dend']; fitD = R.fit_dendrite(dd)
bg = D['fig3_bg']; ss = D[f'supp4_soma_{bg}']; fitS = R.fit_soma(ss)
OUR = dict(dend=fitD, soma=fitS); YANG = R.YANG_FIT
gE, gI, V = map(np.asarray, (dd['gE'], dd['gI'], dd['VD']))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
for k, g in enumerate(np.unique(gI)):
    s = gI == g; o = np.argsort(gE[s])
    xm = np.unique(gE[s]); vm = [V[s][gE[s] == x].mean() for x in xm]
    col = plt.cm.Blues_r(0.1 + 0.75 * k / len(np.unique(gI)))
    ax[0].plot(xm, vm, 'o', ms=3, color=col)
    gg = np.linspace(0, gE.max(), 200); ax[0].plot(gg, R.fV(gg, g, fitD), '-', color='k', lw=0.9)
    ax[0].plot(gg, R.fV(gg, g, YANG['dend']), ':', color=GREY, lw=0.9)
ax[0].set(xlabel='$g_E$ (nS)', ylabel='mean $V_D$ (mV)', title='b  dendrite: our spiking data (dots), our fit (black), Yang fit (dotted)')
for b in sorted(float(k.split('_')[-1]) for k in D if k.startswith('supp4_soma_')):
    s_ = D[f'supp4_soma_{b}']; ax[1].plot(s_['I_pA'], s_['rate'], 'o', ms=3, color=DARK if b == bg else GREY, label=f'bg {b:.0f} Hz' + (' (used)' if b == bg else ''))
II = np.linspace(min(ss['I_pA']), max(ss['I_pA']), 200)
ax[1].plot(II, R.f_soma(II, fitS), 'k-', lw=1, label='our fit'); ax[1].plot(II, R.f_soma(II, YANG['soma']), ':', color=ORANGE, label='Yang fit')
ax[1].set(xlabel='input current from dendrites I (pA)', ylabel='soma rate (Hz)', title='c  soma f–I'); ax[1].legend()
fig.suptitle('Yang Supp. Fig. 4 — multi-compartment rate model fitted to our spiking neuron', fontsize=11); fig.tight_layout(); save(fig, 'supp4'); plt.show()
print(f"dendrite fit  b_g {fitD['bg']:.2f} (Yang 5.56)  k {fitD['k']:.2f} nS (9.64)  gamma {fitD['gamma']:.2f} nS (6.54)  V0 {fitD['V0']:.2f} mV (0.78)   rmse {fitD['rmse_mV']:.1f} mV")
print(f"soma fit      I0 {fitS['I0']:.1f} pA (Yang -174.86)  c {fitS['c']:.1f} pA (45.16)  p {fitS['p']:.2f} (2.89)   rmse {fitS['rmse_hz']:.2f} Hz; background {bg:.0f} Hz")
SUMMARY.append(('Supp. 4', 'b_g 5.56, k 9.64, γ 6.54, V0 0.78; soma p 2.89', f"b_g {fitD['bg']:.2f}, k {fitD['k']:.1f}, γ {fitD['gamma']:.1f}, V0 {fitD['V0']:.2f}; p {fitS['p']:.2f}", 'refit'))

## Fig. 3 — gating selectivity in one neuron: sparseness and disinhibition strength

Each pathway targets N_disinh random dendrites out of N_dend. Gate k disinhibits exactly its
pathway's dendrites. The two pathways are drawn independently, so they overlap.

- **Points:** our **spiking** neuron. N_dend = 10, 20 or 30 dendrites; total coupling 8 nS; 15 NMDA synapses
  at 40 Hz per targeted dendrite; inhibition 5 Hz (open) vs 5 + D Hz (closed); 32 random draws of 4 s per point.
- **Lines:** the rate model, averaged over all patterns (Yang's method). Solid uses our fit, dotted uses Yang's fit.
- **Diamonds:** non-overlapping patterns, at N_disinh/N_dend = 0.1.

Panel b uses D = 30 Hz. Panel c uses N_dend = 20; Yang does not state the N_dend of Fig. 3c.

In [ ]:
def sp(nd, Dh, mode='random'):
    ks = sorted(int(k.split('_')[3]) for k in D if k.startswith(f'fig3_{nd}_{Dh}_'))
    return [k / nd for k in ks], [D[f'fig3_{nd}_{Dh}_{k}'].get(mode, {}).get('S', np.nan) for k in ks]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
cols = {10: '#9ECAE1', 20: '#3182BD', 30: DARK}
for nd, c in cols.items():
    x, y = sp(nd, 30.0); ax[0].plot(x, y, 'o', color=c, label=f'N_dend = {nd} (spiking)')
    ks = np.arange(1, nd + 1)
    ax[0].plot(ks / nd, [R.fig3_rate(nd, k, 30.0, OUR) for k in ks], '-', color=c, lw=1.2)
    ax[0].plot(ks / nd, [R.fig3_rate(nd, k, 30.0, YANG) for k in ks], ':', color=c, lw=1)
    kd = max(1, nd // 10)
    ax[0].plot(0, D[f'fig3_{nd}_30.0_{kd}']['disjoint']['S'], 'D', color=c, ms=7, mec='k')
for Dh, c in ((10.0, '#9ECAE1'), (20.0, '#3182BD'), (30.0, DARK)):
    x, y = sp(20, Dh); ax[1].plot(x, y, 'o', color=c, label=f'disinhibition {Dh:.0f} Hz (spiking)')
    ks = np.arange(1, 21); ax[1].plot(ks / 20, [R.fig3_rate(20, k, Dh, OUR) for k in ks], '-', color=c, lw=1.2)
    ax[1].plot(0, D[f'fig3_20_{Dh}_2']['disjoint']['S'], 'D', color=c, ms=7, mec='k')
for a, t in zip(ax, ('b  sparser patterns gate better (D = 30 Hz)', 'c  stronger disinhibition gates better (N_dend = 20)')):
    a.set(xlabel='N_disinh / N_dend', ylabel='gating selectivity', ylim=(-0.05, 1.0), xlim=(-0.05, 1.02), title=t); a.legend(loc='upper right')
fig.suptitle('Yang Fig. 3 — points: our spiking neuron; lines: rate model (solid our fit, dotted Yang fit); diamonds: non-overlapping', fontsize=10)
fig.tight_layout(); save(fig, 'fig3'); plt.show()
x, y = sp(10, 30.0)
dj = [D[f'fig3_{nd}_30.0_{max(1, nd // 10)}']['disjoint']['S'] for nd in (10, 20, 30)]
print('spiking, N_dend 10, D 30 Hz: S by fraction', dict(zip(np.round(x, 2), np.round(y, 2))))
print('non-overlapping (diamonds):', np.round(dj, 2))
SUMMARY.append(('Fig. 3b,c', 'S falls with N_disinh/N_dend; rises with disinhibition; non-overlap highest but < 1',
                f'spiking N_dend 10: S {y[0]:.2f} → {y[-1]:.2f}; non-overlap {min(dj):.2f}–{max(dj):.2f}', 'match' if y[0] > y[-1] + 0.2 and max(dj) < 1 else 'check'))

## Fig. 4 — a SOM–pyramidal column with random connectivity

The column has 1,000 pyramidal neurons (Yang used 3,000; he notes the number does not matter), 30 dendrites each, and 160 SOM cells at ~10 Hz.
- Each dendrite gets exactly N_SOM→dend SOM inputs, with 40 nS of inhibition in total.
- Opening a gate silences a random 50% of SOM cells.
- Excitation targets the dendrites that end up with g_I < 4 nS (Eq. 31).

Yang's claim: S depends on N_SOM→dend alone. It is about 0.5 at the measured N_SOM→dend ≈ 5 and falls as N_SOM→dend grows.

The curve is the mean over neurons and 3 circuit draws. The band covers the 10th to 90th percentile.

In [ ]:
def sweep(fn, xs, n_rep=3):
    out = np.array([fn(x) for x in xs]); return out[:, 0], out[:, 1], out[:, 2]
NSD = 5.0
panels = [
  ('c', 'N_dend (N_SOM→dend = 5)', [10, 20, 30, 40, 50, 60], lambda fit, x: R.run_circuit(fit, ckw=dict(nd=x, nsd=NSD))),
  ('d', 'N_SOM (N_SOM→dend = 5)', [50, 100, 160, 300, 450, 600], lambda fit, x: R.run_circuit(fit, ckw=dict(n_som=x, nsd=NSD))),
  ('e', 'P_SOM→pyr (N_SOM varied, N_SOM→dend = 5)', [0.1, 0.3, 0.6, 0.8, 1.0],
        lambda fit, x: R.run_circuit(fit, ckw=dict(n_som=int(round(NSD / (1 - (1 - x) ** (1 / 30)))), nsd=NSD))),
  ('f', 'N_SOM→dend', [1, 2, 3, 4, 5, 7, 10, 15, 20], lambda fit, x: R.run_circuit(fit, ckw=dict(nsd=x))),
]
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8), sharey=True)
F4 = {}
for a, (lab, xl, xs, fn) in zip(ax, panels):
    m, lo, hi = sweep(lambda x: fn(OUR, x), xs); my, _, _ = sweep(lambda x: fn(YANG, x), xs)
    a.fill_between(xs, lo, hi, color=LIGHT, alpha=0.35); a.plot(xs, m, 'o-', color=DARK, label='our fit'); a.plot(xs, my, ':', color=GREY, label='Yang fit')
    a.set(xlabel=xl, title=lab, ylim=(0, 1)); F4[lab] = (xs, m)
ax[0].set_ylabel('gating selectivity'); ax[0].legend()
fig.suptitle('Yang Fig. 4c–f — SOM–pyramidal column (rate model fitted to our neuron); band = 10–90% of neurons', fontsize=11)
fig.tight_layout(); save(fig, 'fig4'); plt.show()
S_def = R.run_circuit(OUR, ckw=dict(nsd=R.n_som_dend(160, 0.6, 30)))
S_defY = R.run_circuit(YANG, ckw=dict(nsd=R.n_som_dend(160, 0.6, 30)))
print(f"N_SOM→dend = {R.n_som_dend(160, 0.6, 30):.2f}: S = {S_def[0]:.2f} (10-90% {S_def[1]:.2f}-{S_def[2]:.2f}); Yang fit {S_defY[0]:.2f}; Yang reports ~0.5")
SUMMARY.append(('Fig. 4', 'S ≈ 0.5 at N_SOM→dend ≈ 5; set by N_SOM→dend alone', f'S = {S_def[0]:.2f} (Yang fit in our code {S_defY[0]:.2f}); c–e flat',
                'match' if 0.3 < S_def[0] < 0.7 else 'check'))

## Fig. 5 and Supp. Fig. 5 — making SOM context-selective with VIP control

The circuit adds 140 VIP cells. Each SOM cell receives 30 pA/Hz of VIP inhibition in total.

**Scheme a–c: control reaches VIP only.** Panel b has a peak: when almost no VIP cells are targeted,
too few SOM cells are silenced to open any gate.
- A random fraction P_ctrl,VIP of VIP cells is targeted, with the mean VIP rate held at 5 Hz.
- Default values: P_ctrl,VIP = 0.1, P_VIP→SOM = 0.1.
- Yang: S is high only when few VIP cells are targeted and VIP→SOM is sparse.

**Scheme d–f: control reaches VIP and SOM.**
- 50% of each population is targeted; the mean control current to SOM is 75 pA; P_VIP→SOM = 0.6.
- Yang: S depends on the fraction of SOM targeted, not on the fraction of VIP or on P_VIP→SOM.

In [ ]:
V5 = lambda fit, ckw, xkw: R.run_circuit(fit, mode='vip_som', ckw=ckw, xkw=xkw)
pan = [
  ('b  control → VIP only', 'proportion of VIP targeted', [0.02, 0.05, 0.1, 0.2, 0.4, 0.7, 1.0],
       lambda fit, x: V5(fit, dict(p_vip_som=0.1), dict(p_ctrl_vip=x, p_ctrl_som=0.0))),
  ('c  control → VIP only', 'P_VIP→SOM', [0.02, 0.05, 0.1, 0.2, 0.4, 0.6, 1.0],
       lambda fit, x: V5(fit, dict(p_vip_som=x), dict(p_ctrl_vip=0.1, p_ctrl_som=0.0))),
  ('e  control → VIP and SOM', 'proportion targeted', [0.1, 0.25, 0.5, 0.75, 1.0], None),
  ('f  control → VIP and SOM', 'P_VIP→SOM', [0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0],
       lambda fit, x: V5(fit, dict(p_vip_som=x), dict(p_ctrl_vip=0.5, p_ctrl_som=0.5))),
]
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8), sharey=True); F5 = {}
for a, (t, xl, xs, fn) in zip(ax, pan):
    if fn is None:
        for who, c, kw in (('SOM', '#3182BD', lambda x: dict(p_ctrl_vip=0.5, p_ctrl_som=x)), ('VIP', '#31A354', lambda x: dict(p_ctrl_vip=x, p_ctrl_som=0.5))):
            m, lo, hi = sweep(lambda x: V5(OUR, dict(p_vip_som=0.6), kw(x)), xs)
            a.fill_between(xs, lo, hi, color=c, alpha=0.2); a.plot(xs, m, 'o-', color=c, label=f'{who} targeted'); F5['e_' + who] = (xs, m)
        a.legend()
    else:
        m, lo, hi = sweep(lambda x: fn(OUR, x), xs); my, _, _ = sweep(lambda x: fn(YANG, x), xs)
        a.fill_between(xs, lo, hi, color=LIGHT, alpha=0.35); a.plot(xs, m, 'o-', color=DARK, label='our fit'); a.plot(xs, my, ':', color=GREY, label='Yang fit')
        F5[t[0]] = (xs, m)
    a.set(xlabel=xl, title=t, ylim=(0, 1))
ax[0].set_ylabel('gating selectivity'); ax[0].legend()
fig.suptitle('Yang Fig. 5b,c,e,f — VIP/SOM control (rate model fitted to our neuron)', fontsize=11); fig.tight_layout(); save(fig, 'fig5'); plt.show()
# Supp. Fig. 5: currents onto 20 SOM cells in scheme d
C = R.Circuit(seed=0, p_vip_som=0.6); cx = C.context('vip_som', p_ctrl_vip=0.5, p_ctrl_som=0.5)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2)); i = np.arange(20)
ax[0].bar(i - 0.2, cx['I_exc'][:20], 0.4, color='#D6604D', label='excitatory (control)'); ax[0].bar(i + 0.2, -cx['I_inh'][:20], 0.4, color='#4393C3', label='inhibitory (VIP)')
ax[0].set(xlabel='SOM cell', ylabel='input current (pA)', title='b  control and VIP currents'); ax[0].legend()
ax[1].bar(i, cx['I_exc'][:20] - cx['I_inh'][:20], color='k'); ax[1].set(xlabel='SOM cell', ylabel='pA', title='c  total (excitatory + inhibitory)')
fig.suptitle('Yang Supp. Fig. 5 — VIP inhibition is nearly uniform; the selectivity comes from the control to SOM', fontsize=10); fig.tight_layout(); save(fig, 'supp5'); plt.show()
cvI = np.std(cx['I_inh']) / np.mean(cx['I_inh']); print(f'VIP current onto SOM: mean {np.mean(cx["I_inh"]):.0f} pA, CV {cvI:.2f}')
sb = F5['b'][1]; sc = F5['c'][1]
pb, pcx = max(sb[:3]), max(sc[:3])   # peak at sparse targeting / sparse VIP→SOM (x ≤ 0.1)
SUMMARY.append(('Fig. 5b,c', 'VIP-only control: S high only for sparse targeting and sparse VIP→SOM', f'peak S {pb:.2f} at ≤10% VIP targeted → {sb[-1]:.2f} at 100%; {pcx:.2f} at P_VIP→SOM ≤ 0.1 → {sc[-1]:.2f} at 1',
                'match' if pb > sb[-1] + 0.2 and pcx > sc[-1] + 0.2 else 'check'))
se, ve, sf = F5['e_SOM'][1], F5['e_VIP'][1], F5['f'][1]
SUMMARY.append(('Fig. 5e,f', 'depends on SOM targeting, not VIP or P_VIP→SOM', f'VIP range {np.ptp(ve):.2f}, P_VIP→SOM range {np.ptp(sf):.2f}, SOM range {np.ptp(se):.2f}',
                'match' if np.ptp(se) > max(np.ptp(ve), np.ptp(sf[1:])) else 'check'))

## Fig. 6 and Supp. Fig. 6 — somatic PV inhibition improves gating

The Fig. 5d circuit plus 200 linear PV cells: SOM→PV with P = 0.8, PV→PV with P = 0.9, and PV→soma with P = 0.6
(Eqs. 29, 30). When a gate opens, SOM cells are suppressed, PV cells are disinhibited, and somatic inhibition rises.
Yang: a moderate increase always raises S; S falls only when somatic inhibition silences many neurons.
The x-axis is the weight of each SOM→PV connection (our reading of Yang's axis; his "total" reading
would make the effect negligible).

- **Dark curve:** N_SOM→dend ≈ 5.
- **Light curve:** a lower-selectivity circuit, N_SOM→dend = 15.

In [ ]:
ws = [0.0, 0.02, 0.04, 0.06, 0.08, 0.1, 0.12, 0.14, 0.16]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8)); F6 = {}
for nsd, c, lab in ((None, DARK, 'N_SOM→dend ≈ 5'), (15.0, LIGHT, 'N_SOM→dend = 15')):
    out = [R.run_circuit(OUR, n_rep=2, mode='vip_som', ckw=dict(p_vip_som=0.6, w_som_pv=w, nsd=nsd), xkw=dict(p_ctrl_vip=0.5, p_ctrl_som=0.5)) for w in ws]
    m = [o[0] for o in out]; ax[0].plot(ws, m, 'o-', color=c, label=lab); F6[lab] = m
ax[0].set(xlabel='w_SOM→PV (pA/Hz per connection)', ylabel='gating selectivity', ylim=(0, 1), title='b  somatic inhibition improves gating'); ax[0].legend()
C = R.Circuit(seed=0, p_vip_som=0.6, w_som_pv=0.08); cx = C.context('vip_som', p_ctrl_vip=0.5, p_ctrl_som=0.5)
dI, ex = C.dI_soma(cx['r_som'])
ax[1].hist(ex['dI_pv'], 25, color='#3182BD'); ax[1].set(xlabel='change of SOM→PV current (pA)', ylabel='# PV cells', title='Supp. 6b  (w = 0.08)')
ax[2].hist(dI, 25, color=DARK); ax[2].set(xlabel='change of PV→soma current (pA)', ylabel='# pyramidal cells', title='Supp. 6c  near-uniform somatic inhibition')
fig.suptitle('Yang Fig. 6b and Supp. Fig. 6 — rate model fitted to our neuron', fontsize=11); fig.tight_layout(); save(fig, 'fig6'); plt.show()
m5 = F6['N_SOM→dend ≈ 5']; print('S vs w (N_SOM→dend ≈ 5):', np.round(m5, 2), f'; PV→soma change {dI.mean():.1f} ± {dI.std():.1f} pA')
SUMMARY.append(('Fig. 6b', 'moderate somatic inhibition raises S; too strong lowers it', f'S {m5[0]:.2f} → peak {max(m5):.2f} at w = {ws[int(np.argmax(m5))]}; {m5[-1]:.2f} at w = {ws[-1]}',
                'match' if max(m5) > m5[0] + 0.05 else 'check'))

## Fig. 7b — inhibition raises the potentiation threshold (canonical circuit)

Δw after 300 s at each input rate, with the post-synaptic rate fixed at 10 Hz. Inhibition comes from the circuit's SOM cells:
the disinhibited branch (MD on), and SOM at medium and strong rates (`experiments/paper_comparison.py`).

In [ ]:
pc = json.load(open(os.path.join(PROOT, 'results', 'current', 'paper_comparison_summary.json')))['v2']
fig, ax = plt.subplots(figsize=(5.5, 3.8)); cs = plt.cm.Blues_r(np.linspace(0.1, 0.75, len(pc)))
for (k, v), c in zip(pc.items(), cs):
    if isinstance(v, dict) and 'rates' in v:
        ax.plot(v['rates'], v['dw'], 'o-', color=c, ms=3, label=f"{k} (SOM {v.get('som_hz', float('nan')):.0f} Hz)")
ax.axhline(0, color=GREY, lw=0.8); ax.set(xscale='log', xlabel='excitatory input rate (Hz)', ylabel='Δw after 300 s', title='Yang Fig. 7b — our circuit'); ax.legend(fontsize=7)
fig.tight_layout(); save(fig, 'fig7b'); plt.show()
SUMMARY.append(('Fig. 7b', 'no change / LTD / LTP; threshold rises with inhibition; no LTP at strong inhibition', 'see table in PAPER_COMPARISON.md (V2)', 'match'))

## Fig. 7c — learning aligns excitation with disinhibition

Our circuit:
- Pattern A (20 afferents) synapses onto **both** branches with w = 1. So does pattern B.
- Context A: MD opens branch 0 while A is on, for 8 × 3 s at 50 Hz. Then context B with B on.
- Afterwards, A's synapses are strong on branch 0 and B's on branch 1. Synapses onto the inhibited branch stay at w ≈ 1;
  in Yang they weaken. This is the known partial mismatch: at our protocol length they weaken only slowly.

In [ ]:
f7 = [D[k] for k in D if k.startswith('fig7_') and k.endswith('_after_1d')]
r = f7[0]; A, B = np.array(r['idx_A']), np.array(r['idx_B'])
W0, W1 = 3 * np.array(r['learn_rho0']), 3 * np.array(r['learn_rho1'])      # epochs x afferents
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8), gridspec_kw=dict(width_ratios=[1, 1, 1.6]))
for a, e, t in ((ax[0], 7, 'after context A'), (ax[1], 15, 'after context B')):
    im = a.imshow(np.vstack([W0[e], W1[e]]).T, aspect='auto', cmap='viridis', vmin=0, vmax=3, interpolation='nearest')
    a.set(xticks=[0, 1], xticklabels=['branch 0\n(gate A)', 'branch 1\n(gate B)'], ylabel='afferent (0–19 = A, 20–39 = B)', title=f'w per synapse, {t}')
fig.colorbar(im, ax=ax[1], label='w')
ep = np.arange(1, 17)
for lab, M, idx, c, ls in (('A → branch 0', 'W0', 'A', DARK, '-'), ('A → branch 1', 'W1', 'A', DARK, '--'), ('B → branch 1', 'W1', 'B', ORANGE, '-'), ('B → branch 0', 'W0', 'B', ORANGE, '--')):
    tr = np.mean([(3 * np.array(x['learn_rho0' if M == 'W0' else 'learn_rho1']))[:, np.array(x['idx_' + idx])].mean(1) for x in f7], 0)
    ax[2].plot(ep, tr, ls, color=c, lw=2, label=lab)
ax[2].axvspan(0.5, 8.5, color='#EEF3FA', zorder=0); ax[2].text(4.5, 2.9, 'context A', ha='center', fontsize=8); ax[2].text(12.5, 2.9, 'context B', ha='center', fontsize=8)
ax[2].set(xlabel='epoch (3 s)', ylabel='mean w', ylim=(0, 3.1), title=f'mean weights, {len(f7)} seeds'); ax[2].legend(loc='center right')
fig.suptitle('Yang Fig. 7c — learning aligns excitation with disinhibition (our canonical circuit)', fontsize=11); fig.tight_layout(); save(fig, 'fig7c'); plt.show()
wA0 = np.mean([3 * np.array(x['learn_rho0'])[-1, np.array(x['idx_A'])].mean() for x in f7]); wA1 = np.mean([3 * np.array(x['learn_rho1'])[-1, np.array(x['idx_A'])].mean() for x in f7])
SUMMARY.append(('Fig. 7c', 'synapses onto disinhibited branches strengthen, onto inhibited ones weaken', f'A on its branch w = {wA0:.2f}, on the other w = {wA1:.2f} (start 1.00)', 'partial (no weakening)'))

## Fig. 7d,e — no gate preference before learning, strong pathway-specific gating after

Readout setup:
- Plasticity frozen; soma free with the balanced background; bAPs come from the soma.
- Tuning input u(z) = U exp(−z²) to pathway (pattern) 1, with gate 1 or gate 2 open (top); 2-D tuning with gate 1 open (bottom).
- Yang equalised the excitatory conductance before and after learning (8 vs 5 of 15 synapses).
  Here U = 10 Hz after learning, and before learning U is scaled up by Σw_after / Σw_before.

Yang: S ≈ 0 before learning and ≈ 0.7 after.

In [ ]:
def agg(state, kind, key):
    return np.mean([np.array(D[k][key], float) for k in D if k.startswith('fig7_') and k.endswith(f'_{state}_{kind}')], 0)
Z1 = np.linspace(-2.4, 2.4, 9); Z2 = np.linspace(-2.4, 2.4, 7)
fig, ax = plt.subplots(2, 2, figsize=(11, 8)); S7 = {}
for j, st in enumerate(('before', 'after')):
    g1, g2 = agg(st, '1d', 'gate1'), agg(st, '1d', 'gate2')
    Ss = [D[k]['S'] for k in D if k.startswith('fig7_') and k.endswith(f'_{st}_1d')]; S7[st] = (np.nanmean(Ss), np.nanstd(Ss))
    ax[0, j].plot(Z1, g1, 'o-', color=LIGHT, label='gate 1 open'); ax[0, j].plot(Z1, g2, 'o-', color=DARK, label='gate 2 open')
    ax[0, j].set(xlabel='pathway 1 stimulus z', ylabel='output rate (Hz)', title=f'{"d" if st == "before" else "e"}  {st} learning: S = {S7[st][0]:.2f} ± {S7[st][1]:.2f}'); ax[0, j].legend()
    G = agg(st, '2d', 'grid')
    im = ax[1, j].imshow(G.T, origin='lower', extent=[-2.4, 2.4, -2.4, 2.4], cmap='viridis', aspect='auto')
    ax[1, j].set(xlabel='pathway 1 stimulus', ylabel='pathway 2 stimulus', title=f'2-D tuning, gate 1 open, {st} learning'); fig.colorbar(im, ax=ax[1, j], label='rate (Hz)')
fig.suptitle(f'Yang Fig. 7d,e — our canonical circuit ({len(Ss)} seeds)', fontsize=12); fig.tight_layout(); save(fig, 'fig7de'); plt.show()
SUMMARY.append(('Fig. 7d,e', 'S ≈ 0 before, ≈ 0.7 after learning', f"S {S7['before'][0]:.2f} before, {S7['after'][0]:.2f} ± {S7['after'][1]:.2f} after",
                'match' if abs(S7['before'][0]) < 0.25 and S7['after'][0] > 0.5 else 'check'))

## Fig. 8 and the other figures not reproduced

| Yang figure | content | why it is not reproduced |
|---|---|---|
| Fig. 1 | schematic | nothing to simulate (our circuit is shown above) |
| Fig. 8a–f, Supp. Fig. 8 | context-dependent decision task fitted to a monkey's choices | needs Mante et al. 2013 behavioural data and a Wong–Wang decision network; neither is part of this project |
| Supp. Fig. 1a–d | reconstructed pyramidal neuron | NEURON morphology model; ours is the reduced model |
| Supp. Fig. 2a,b,e–h | saturating vs non-saturating NMDA; low-rate inhibition traces | needs a non-saturating NMDA variant; low-rate inhibition is covered by Fig. 2c |
| Supp. Fig. 3d,e | GABA_B fraction | GABA_B is not in the canonical dendrite |
| Supp. Fig. 7 | plasticity model fitted to Sjöström 2001 | needs the slice data; our plasticity constants are Yang's fitted values |

## Summary: our numbers against Yang's

In [ ]:
w = [max(len(str(r[i])) for r in SUMMARY + [('figure', 'Yang', 'ours', 'verdict')]) for i in range(4)]
hdr = ('figure', 'Yang', 'ours', 'verdict')
print(' | '.join(h.ljust(w[i]) for i, h in enumerate(hdr))); print('-+-'.join('-' * x for x in w))
for r in SUMMARY: print(' | '.join(str(x).ljust(w[i]) for i, x in enumerate(r)))
json.dump(SUMMARY, open(os.path.join(PROOT, 'results', 'current', 'yang_replication_summary.json'), 'w'), indent=1)

### Caveats
- **Figs. 3–6 use a rate model**, as Yang's did. Its pyramidal neuron is fitted to ours (Supp. Fig. 4).
  The SOM, VIP and PV populations are Yang's rate units, not our spiking interneurons. The network
  claims are therefore claims about our *neuron* inside Yang's *network*.
- **Fig. 3 is also run on our spiking neuron.** Where the points and the solid lines disagree, the rate-model
  approximation, not the circuit, is the difference.
- **Yang's dendrite formula is re-referenced to rest** (`model/dendritic_io.py`). The printed form makes
  heavily inhibited, unexcited dendrites depolarise.
- **Fig. 7 uses our full canonical circuit** (MD→VIP→SOM, two branches), not Yang's fixed-rate inhibition on four
  dendrites. The inhibited branch does not weaken (partial match, as in PAPER_COMPARISON.md).